# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jh-emon002/flyrank-intern/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn

import os
import numpy as np
import pandas as pd
import duckdb
import sklearn

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

from sklearn.inspection import permutation_importance

RANDOM_STATE = 42

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

assert HF_TOKEN, "HF_TOKEN not found."

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/"
    f"month=2026-03/*.parquet'"
    f")"
)

FEATURE_START = "2026-03-01"
FEATURE_END = "2026-03-15"

EARLY_START = "2026-03-01"
EARLY_END = "2026-03-07"

RECENT_START = "2026-03-09"
RECENT_END = "2026-03-15"

OUTCOME_START = "2026-03-17"
OUTCOME_END = "2026-03-31"

MIN_IMPRESSIONS = 100
DECLINE_THRESHOLD = 0.80

print("sklearn:", sklearn.__version__)
print("Setup complete.")

sklearn: 1.6.1
Setup complete.


## 1. Method choice and why

My outcome is binary: whether a content item experiences a future impression decline `is_declining_next15d`. However, the operational objective is ranking: pages with the highest estimated risk should appear first in a review queue.

I therefore start with Logistic Regression. It provides a simple, interpretable probabilistic baseline and allows me to test whether multiple safe pre-decision signals improve on my Week-4 hand-written score.

I also test a constrained Random Forest as a nonlinear challenger. It can capture interactions and nonlinear relationships that Logistic Regression cannot, but I will prefer the simpler model unless the Random Forest produces a meaningful improvement on the same held-out data.

Both models use only features available before the March 16 decision point. Model probability is used as the ranking score.

In [2]:
df = con.sql(f"""
WITH page_windows AS (

    SELECT
        client_hash_id,
        content_hash_id,

        COUNT(DISTINCT CASE
            WHEN report_date BETWEEN DATE '{FEATURE_START}'
                                 AND DATE '{FEATURE_END}'
             AND gsc_data_available IS TRUE
            THEN report_date
        END) AS feature_days_available,

        COUNT(DISTINCT CASE
            WHEN report_date BETWEEN DATE '{OUTCOME_START}'
                                 AND DATE '{OUTCOME_END}'
             AND gsc_data_available IS TRUE
            THEN report_date
        END) AS outcome_days_available,

        SUM(CASE
            WHEN report_date BETWEEN DATE '{FEATURE_START}'
                                 AND DATE '{FEATURE_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_impressions
            ELSE 0
        END) AS impressions_pre15,

        -- Early 7 days
        SUM(CASE
            WHEN report_date BETWEEN DATE '{EARLY_START}'
                                 AND DATE '{EARLY_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_impressions
            ELSE 0
        END) AS impressions_early7,

        SUM(CASE
            WHEN report_date BETWEEN DATE '{EARLY_START}'
                                 AND DATE '{EARLY_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_clicks
            ELSE 0
        END) AS clicks_early7,

        SUM(CASE
            WHEN report_date BETWEEN DATE '{EARLY_START}'
                                 AND DATE '{EARLY_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_avg_position * gsc_impressions
            ELSE 0
        END)
        /
        NULLIF(
            SUM(CASE
                WHEN report_date BETWEEN DATE '{EARLY_START}'
                                     AND DATE '{EARLY_END}'
                 AND gsc_data_available IS TRUE
                THEN gsc_impressions
                ELSE 0
            END),
            0
        ) AS avg_position_early7,

        -- Recent 7 days
        SUM(CASE
            WHEN report_date BETWEEN DATE '{RECENT_START}'
                                 AND DATE '{RECENT_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_impressions
            ELSE 0
        END) AS impressions_recent7,

        SUM(CASE
            WHEN report_date BETWEEN DATE '{RECENT_START}'
                                 AND DATE '{RECENT_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_clicks
            ELSE 0
        END) AS clicks_recent7,

        SUM(CASE
            WHEN report_date BETWEEN DATE '{RECENT_START}'
                                 AND DATE '{RECENT_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_avg_position * gsc_impressions
            ELSE 0
        END)
        /
        NULLIF(
            SUM(CASE
                WHEN report_date BETWEEN DATE '{RECENT_START}'
                                     AND DATE '{RECENT_END}'
                 AND gsc_data_available IS TRUE
                THEN gsc_impressions
                ELSE 0
            END),
            0
        ) AS avg_position_recent7,

        -- FUTURE: label construction only
        SUM(CASE
            WHEN report_date BETWEEN DATE '{OUTCOME_START}'
                                 AND DATE '{OUTCOME_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_impressions
            ELSE 0
        END) AS impressions_next15

    FROM {MARCH}

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT *
FROM page_windows
WHERE
    feature_days_available = 15
    AND outcome_days_available = 15
    AND impressions_pre15 >= {MIN_IMPRESSIONS}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
df["recent_trend_pct"] = (
    100.0
    * (df["impressions_recent7"] - df["impressions_early7"])
    / df["impressions_early7"].replace(0, np.nan)
)

df["ctr_early7_pct"] = (
    100.0
    * df["clicks_early7"]
    / df["impressions_early7"].replace(0, np.nan)
)

df["ctr_recent7_pct"] = (
    100.0
    * df["clicks_recent7"]
    / df["impressions_recent7"].replace(0, np.nan)
)

# Positive = ranking position became worse
df["position_change"] = (
    df["avg_position_recent7"]
    - df["avg_position_early7"]
)

df["log_impressions_early7"] = np.log1p(
    df["impressions_early7"]
)

df["log_impressions_recent7"] = np.log1p(
    df["impressions_recent7"]
)

# FUTURE LABEL
df["decline_ratio"] = (
    df["impressions_next15"]
    / df["impressions_pre15"]
)

df["is_declining_next15d"] = (
    df["decline_ratio"] < DECLINE_THRESHOLD
).astype(int)

df = df.replace([np.inf, -np.inf], np.nan)

print("Eligible pages:", len(df))
print(
    "Overall decline base rate:",
    round(df["is_declining_next15d"].mean(), 3)
)

Eligible pages: 58097
Overall decline base rate: 0.345


## 2. Split design

I use a client-grouped holdout split. All pages belonging to one client are assigned entirely to either training or test data.

This tests whether the model generalizes to unseen clients rather than merely learning patterns specific to clients represented during training. `client_hash_id` and `content_hash_id` are never model features.

I use a fixed `random seed (42)` for reproducibility. The Week-4 baseline and both learned models are evaluated on exactly the same held-out test pages.

In [4]:
TARGET = "is_declining_next15d"

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    splitter.split(
        df,
        y=df[TARGET],
        groups=df["client_hash_id"]
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

train_clients = set(train_df["client_hash_id"])
test_clients = set(test_df["client_hash_id"])

assert train_clients.isdisjoint(test_clients)

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))

print(
    "Train base rate:",
    round(train_df[TARGET].mean(), 3)
)

print(
    "Test base rate:",
    round(test_df[TARGET].mean(), 3)
)

print(
    "Client overlap:",
    len(train_clients & test_clients)
)

Train rows: 14350
Test rows: 43747
Train clients: 26
Test clients: 7
Train base rate: 0.296
Test base rate: 0.361
Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [5]:
FEATURES = [
    "log_impressions_early7",
    "log_impressions_recent7",
    "recent_trend_pct",
    "ctr_early7_pct",
    "ctr_recent7_pct",
    "avg_position_early7",
    "avg_position_recent7",
    "position_change"
]


In [6]:
def add_baseline_score(frame):

    out = frame.copy()

    decline_strength = (
        -out["recent_trend_pct"] / 100.0
    ).clip(lower=0, upper=1).fillna(0)

    trigger = (
        (out["recent_trend_pct"] <= -20.0)
        & (out["impressions_recent7"] > 0)
    )

    out["baseline_score"] = np.where(
        trigger,
        np.log1p(out["impressions_recent7"])
        * decline_strength,
        0.0
    )

    return out


test_df = add_baseline_score(test_df)

In [7]:
X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]

logreg = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            random_state=RANDOM_STATE
        )
    )
])

logreg.fit(X_train, y_train)

test_df["logreg_score"] = (
    logreg.predict_proba(X_test)[:, 1]
)

print("Logistic Regression trained.")

Logistic Regression trained.


In [8]:
rf = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=20,
            max_features="sqrt",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )
])

rf.fit(X_train, y_train)

test_df["rf_score"] = (
    rf.predict_proba(X_test)[:, 1]
)

print("Random Forest trained.")

Random Forest trained.


In [9]:
def precision_at_k(y_true, score, k):

    evaluation = pd.DataFrame({
        "y": np.asarray(y_true),
        "score": np.asarray(score)
    })

    evaluation = evaluation.sort_values(
        "score",
        ascending=False
    )

    k = min(k, len(evaluation))

    return evaluation.head(k)["y"].mean()


def evaluate_model(name, y_true, score):

    return {
        "method": name,
        "roc_auc": roc_auc_score(y_true, score),
        "average_precision": average_precision_score(
            y_true,
            score
        ),
        "precision_at_10": precision_at_k(
            y_true, score, 10
        ),
        "precision_at_20": precision_at_k(
            y_true, score, 20
        ),
        "precision_at_50": precision_at_k(
            y_true, score, 50
        ),
        "precision_at_100": precision_at_k(
            y_true, score, 100
        )
    }

In [10]:
results = []

results.append(
    evaluate_model(
        "Week-4 baseline",
        y_test,
        test_df["baseline_score"]
    )
)

results.append(
    evaluate_model(
        "Logistic Regression",
        y_test,
        test_df["logreg_score"]
    )
)

results.append(
    evaluate_model(
        "Random Forest",
        y_test,
        test_df["rf_score"]
    )
)

comparison = pd.DataFrame(results)

metric_cols = [
    "roc_auc",
    "average_precision",
    "precision_at_10",
    "precision_at_20",
    "precision_at_50",
    "precision_at_100"
]

comparison[metric_cols] = (
    comparison[metric_cols].round(3)
)

display(comparison)

print(
    "Held-out base rate:",
    round(y_test.mean(), 3)
)

,method,roc_auc,average_precision,precision_at_10,precision_at_20,precision_at_50,precision_at_100
0,Week-4 baseline,0.602,0.455,0.9,0.85,0.78,0.70
1,Logistic Regression,0.666,0.531,1.0,0.90,0.88,0.87
2,Random Forest,0.682,0.543,0.8,0.85,0.78,0.75


Held-out base rate: 0.361


### Model comparison

Both learned models improved on the Week-4 baseline in some respects, but Logistic Regression produced the strongest operational ranking.

The held-out test set had a future-decline base rate of 36.1%. The Week-4 rule achieved Precision@10, @20, @50, and @100 of 0.90, 0.85, 0.78, and 0.70 respectively. Logistic Regression improved these values to 1.00, 0.90, 0.88, and 0.87.

Random Forest achieved the highest ROC-AUC (0.682) and average precision (0.543), slightly above Logistic Regression at 0.666 and 0.531. However, Random Forest performed worse in the highest-priority portion of the queue, with Precision@10 = 0.80 and Precision@100 = 0.75.

Because the practical objective is to rank a limited number of pages for manual review rather than maximize global discrimination alone, I select Logistic Regression as the preferred model. It outperforms both the Week-4 baseline and Random Forest at every reported Precision@K while remaining simpler and easier to interpret. The additional complexity of Random Forest is therefore not justified for this decision problem.


In [11]:
chosen_model = logreg
chosen_score_col = "logreg_score"
chosen_name = "Logistic Regression"

In [12]:
perm = permutation_importance(
    chosen_model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "feature": FEATURES,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values(
    "importance_mean",
    ascending=False
)

display(importance_df)

,feature,importance_mean,importance_std
0,log_impressions_early7,0.164381,0.000838
1,log_impressions_recent7,0.130628,0.000676
3,ctr_early7_pct,0.015540,0.000980
6,avg_position_recent7,0.010739,0.001329
4,ctr_recent7_pct,0.005609,0.000634
5,avg_position_early7,0.005231,0.000836
7,position_change,0.002306,0.000352
2,recent_trend_pct,0.002015,0.000328


In [14]:
coef_df = pd.DataFrame({
    "feature": FEATURES,
    "coefficient": logreg.named_steps["model"].coef_[0]
})

coef_df["abs_coefficient"] = (
    coef_df["coefficient"].abs()
)

coef_df = coef_df.sort_values(
    "abs_coefficient",
    ascending=False
)

display(coef_df)

,feature,coefficient,abs_coefficient
0,log_impressions_early7,1.013553,1.013553
1,log_impressions_recent7,-0.899356,0.899356
3,ctr_early7_pct,-0.213554,0.213554
4,ctr_recent7_pct,-0.090870,0.090870
6,avg_position_recent7,0.083575,0.083575
2,recent_trend_pct,0.062028,0.062028
5,avg_position_early7,0.060503,0.060503
7,position_change,0.052240,0.052240


In [15]:
analysis_df = (
    test_df
    .sort_values(
        "logreg_score",
        ascending=False
    )
    .reset_index(drop=True)
)

analysis_df["model_rank"] = (
    np.arange(len(analysis_df)) + 1
)

analysis_df["selected_top100"] = (
    analysis_df["model_rank"] <= 100
)

In [17]:
false_positives = analysis_df[
    (analysis_df["selected_top100"])
    & (analysis_df[TARGET] == 0)
].copy()

print(
    "False positives in top 100:",
    len(false_positives)
)

display(
    false_positives[
        [
            "model_rank",
            "content_hash_id",
            "logreg_score",
            "impressions_early7",
            "impressions_recent7",
            "recent_trend_pct",
            "ctr_recent7_pct",
            "avg_position_recent7",
            "position_change",
            TARGET
        ]
    ].head(15)
)

False positives in top 100: 13


,model_rank,content_hash_id,logreg_score,impressions_early7,impressions_recent7,recent_trend_pct,ctr_recent7_pct,avg_position_recent7,position_change,is_declining_next15d
14,15,content_b154f6c2652cfeb9,0.812391,3095.0,248.0,-91.987076,0.0,22.479839,21.017157,0
18,19,content_ac671311da8857d9,0.799235,2046.0,144.0,-92.961877,0.0,13.027778,10.162675,0
27,28,content_ca4840c2c8da1946,0.777926,448.0,24.0,-94.642857,0.0,2.125000,-3.622768,0
32,33,content_da0893d91badaac5,0.760619,1028.0,86.0,-91.634241,0.0,10.674419,9.487648,0
33,34,content_eb2d4a43ccac2437,0.758417,976.0,72.0,-92.622951,0.0,4.027778,1.546220,0
43,44,content_8934f9965407083a,0.743602,385.0,91.0,-76.363636,0.0,68.351648,53.868531,0
54,55,content_588f7d5831396d54,0.729389,428.0,53.0,-87.616822,0.0,26.188679,23.450361,0
57,58,content_135ef063501d5a6a,0.726493,1576.0,158.0,-89.974619,0.0,13.974684,7.892831,0
73,74,content_f681c78df0dba8b4,0.703085,393.0,97.0,-75.318066,0.0,58.030928,37.076729,0
78,79,content_90b26f2ec5592253,0.692553,724.0,75.0,-89.640884,0.0,0.653333,-1.874291,0


In [18]:
missed_positives = analysis_df[
    (~analysis_df["selected_top100"])
    & (analysis_df[TARGET] == 1)
].copy()

display(
    missed_positives[
        [
            "model_rank",
            "content_hash_id",
            "logreg_score",
            "impressions_early7",
            "impressions_recent7",
            "recent_trend_pct",
            "ctr_recent7_pct",
            "avg_position_recent7",
            "position_change",
            TARGET
        ]
    ].head(10)
)

,model_rank,content_hash_id,logreg_score,impressions_early7,impressions_recent7,recent_trend_pct,ctr_recent7_pct,avg_position_recent7,position_change,is_declining_next15d
100,101,content_e261c22e04baa4f2,0.668444,330.0,55.0,-83.333333,0.000000,23.000000,20.775758,1
102,103,content_2847fc9e2cf0913a,0.667770,2981.0,460.0,-84.568937,0.000000,6.295652,-0.199819,1
103,104,content_cd83de7ef76067ea,0.667486,648.0,95.0,-85.339506,0.000000,11.421053,8.728151,1
104,105,content_bb1a4495f0f7f69b,0.667120,5862.0,841.0,-85.653361,0.118906,0.658740,-5.203253,1
105,106,content_13f28a7f60bdf8c7,0.666855,1162.0,206.0,-82.271945,0.000000,18.737864,9.739585,1
106,107,content_b686efefc3185a87,0.666397,1067.0,148.0,-86.129335,0.000000,5.459459,0.166114,1
107,108,content_80f45eea2091ac8b,0.665874,842.0,124.0,-85.273159,0.806452,17.483871,13.805724,1
111,112,content_3b4d3f68e1b6c9c6,0.663548,1013.0,147.0,-85.488648,0.000000,9.034014,4.941220,1
113,114,content_c9a5173595d24e37,0.662260,2420.0,385.0,-84.090909,0.000000,5.880519,0.332172,1
114,115,content_715c744ad187d201,0.662099,4131.0,609.0,-85.257807,0.821018,5.313629,3.743065,1


In [19]:
near_miss_positives = analysis_df[
    (analysis_df["model_rank"] > 100)
    & (analysis_df["model_rank"] <= 150)
    & (analysis_df[TARGET] == 1)
]

display(
    near_miss_positives[
        [
            "model_rank",
            "content_hash_id",
            "logreg_score",
            "impressions_early7",
            "impressions_recent7",
            "recent_trend_pct",
            "ctr_recent7_pct",
            "avg_position_recent7",
            "position_change",
            TARGET
        ]
    ]
)

,model_rank,content_hash_id,logreg_score,impressions_early7,impressions_recent7,recent_trend_pct,ctr_recent7_pct,avg_position_recent7,position_change,is_declining_next15d
100,101,content_e261c22e04baa4f2,0.668444,330.0,55.0,-83.333333,0.000000,23.000000,20.775758,1
102,103,content_2847fc9e2cf0913a,0.667770,2981.0,460.0,-84.568937,0.000000,6.295652,-0.199819,1
103,104,content_cd83de7ef76067ea,0.667486,648.0,95.0,-85.339506,0.000000,11.421053,8.728151,1
104,105,content_bb1a4495f0f7f69b,0.667120,5862.0,841.0,-85.653361,0.118906,0.658740,-5.203253,1
105,106,content_13f28a7f60bdf8c7,0.666855,1162.0,206.0,-82.271945,0.000000,18.737864,9.739585,1
106,107,content_b686efefc3185a87,0.666397,1067.0,148.0,-86.129335,0.000000,5.459459,0.166114,1
107,108,content_80f45eea2091ac8b,0.665874,842.0,124.0,-85.273159,0.806452,17.483871,13.805724,1
111,112,content_3b4d3f68e1b6c9c6,0.663548,1013.0,147.0,-85.488648,0.000000,9.034014,4.941220,1
113,114,content_c9a5173595d24e37,0.662260,2420.0,385.0,-84.090909,0.000000,5.880519,0.332172,1
114,115,content_715c744ad187d201,0.662099,4131.0,609.0,-85.257807,0.821018,5.313629,3.743065,1


## 4. Errors and interpretation

At an operational review budget of 100 pages, Logistic Regression produced 13 false positives, consistent with its Precision@100 of 0.87. These errors show a clear pattern rather than appearing random.

All 13 false positives had zero CTR in the recent seven-day window and experienced substantial pre-decision impression declines, ranging approximately from 75% to 95%. Most also showed worsening average search position. These combinations closely resemble genuine persistent decline, so the model assigns them high probabilities.

However, these pages did not meet the future-decline definition during the outcome window. This suggests that an important failure mode is temporary deterioration: the model observes a sharp loss of visibility before the decision point but cannot know whether the page will remain weak or subsequently recover.

Some false positives also have very low recent impression volumes. At low volume, percentage changes can become unstable; relatively small absolute changes can generate extremely large decline percentages. This may cause the model to overstate the importance of short-term deterioration for low-volume pages.

For example, content_b154f6c2652cfeb9 fell from 3,095 to 248 impressions (-92.0%), had zero recent CTR, and its average position worsened by approximately 21 positions. It therefore looked like a strong decline candidate and received a predicted probability of 0.812, but the future label was negative. Similarly, content_ca4840c2c8da1946 fell from 448 to only 24 impressions (-94.6%) but did not subsequently meet the decline definition. These cases illustrate that strong recent deterioration is predictive but not always persistent.

Combined with the near-miss analysis, the errors indicate that the model is strongest at identifying deterioration patterns but has difficulty distinguishing persistent decline from temporary volatility or recovery. Additional historical context, such as a longer pre-decision trend window or measures of traffic volatility, could potentially help distinguish these cases, provided those features remain available before the prediction cutoff.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.